In [ ]:
# Cell 1: Clone code (Kaggle: bật Internet trong Settings). Nhánh phải đã push code mới nhất
!git clone --single-branch --branch HoangMinh https://github.com/VanHung-05/zalo_challenge_2026_datamate.git
%cd zalo_challenge_2026_datamate
!git log --oneline -1

In [ ]:
# Cell 2: Cài thư viện, trừ llama-cpp-python (cài bản CUDA ở cell sau, tránh build bản CPU vô ích)
# Dùng sys.executable để pip cài vào ĐÚNG Python mà các cell sau dùng để chạy runner
import sys
!grep -v '^llama-cpp-python' requirements.txt > /tmp/requirements_no_llama.txt
!{sys.executable} -m pip install -q -r /tmp/requirements_no_llama.txt
!{sys.executable} -m pip install -q diskcache jinja2 typing-extensions  # dependencies của llama-cpp-python (cell sau dùng --no-deps)

In [ ]:
# Cell 2b: Cài llama-cpp-python — có GPU thì build CUDA (compile ~10-20 phút, chạy 1 lần), không có thì bản CPU
# Log build đầy đủ ghi vào /kaggle/working/llama_build.log; build hỏng thì cell DỪNG và in lỗi (không âm thầm chạy CPU)
import os, shutil, subprocess, sys

PY = sys.executable
LOG = "/kaggle/working/llama_build.log"
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
env = dict(os.environ)

if has_gpu:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
    cuda_home = "/usr/local/cuda"
    env["PATH"] = f"{cuda_home}/bin:" + env.get("PATH", "")
    env["CUDACXX"] = f"{cuda_home}/bin/nvcc"
    cmake_args = "-DGGML_CUDA=on"
    # Build chỉ cho kiến trúc GPU đang có (T4 = 7.5 -> 75), nhanh hơn build mọi kiến trúc
    cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], capture_output=True, text=True)
    if cap.returncode == 0 and cap.stdout.strip():
        cmake_args += " -DCMAKE_CUDA_ARCHITECTURES=" + cap.stdout.split()[0].replace(".", "")
    env["CMAKE_ARGS"] = cmake_args
    print("CMAKE_ARGS =", cmake_args, "| nvcc:", shutil.which("nvcc", path=env["PATH"]))
    cmd = [PY, "-m", "pip", "install", "llama-cpp-python", "--force-reinstall", "--no-deps", "--no-cache-dir", "--verbose"]
else:
    print("ℹ️ Không có GPU — cài bản CPU")
    cmd = [PY, "-m", "pip", "install", "llama-cpp-python"]

with open(LOG, "w") as f:
    rc = subprocess.run(cmd, env=env, stdout=f, stderr=subprocess.STDOUT).returncode
if rc != 0:
    print(open(LOG).read()[-6000:])
    raise RuntimeError(f"Cài llama-cpp-python thất bại (exit {rc}), xem {LOG}")

check = subprocess.run([PY, "-c", "import llama_cpp; print(llama_cpp.__version__, llama_cpp.llama_supports_gpu_offload())"],
                       capture_output=True, text=True)
print(check.stdout, check.stderr)
if check.returncode != 0:
    raise RuntimeError("Đã cài nhưng import llama_cpp lỗi, xem stderr ở trên")
if has_gpu and not check.stdout.strip().endswith("True"):
    raise RuntimeError(f"Có GPU nhưng bản build không hỗ trợ GPU offload, xem {LOG}")
print("✅ llama-cpp-python sẵn sàng, chạy trên", "GPU" if has_gpu else "CPU")

## Download model

In [ ]:
!{sys.executable} scripts/download_model.py

## Test inference

In [ ]:
# Chạy thử 5 câu train; đầu output phải có "[LLM] llama.cpp device: GPU". Xem thời gian/câu (elapsed_seconds) trước khi chạy cả bộ
!{sys.executable} src/datamate/runner.py -c configs/dev.yaml -d train --limit 5 -o outputs/train_5/submission.jsonl
!{sys.executable} scripts/evaluate.py -p outputs/train_5/submission.jsonl -g train/answers.jsonl --only-predicted

In [ ]:
# Xem log 1 câu: SQL model sinh ra, lỗi, kết quả
import glob, json
log = json.load(open(sorted(glob.glob('outputs/train_5/logs/*.json'))[0], encoding='utf-8'))
print(log['question'], '->', log['final'], f"({log['elapsed_seconds']}s)")
for a in log.get('attempts', []):
    print('--- attempt', a['attempt'], '| error:', a.get('error'))
    print(a.get('sql'))

## Inference public test

In [ ]:
# Chạy toàn bộ public test (1000 câu), ghi từng dòng nên dừng giữa chừng vẫn giữ kết quả đã chạy
!{sys.executable} src/datamate/runner.py -c configs/dev.yaml -d public_test -o outputs/public/submission.jsonl
!{sys.executable} scripts/validate_submission.py -p outputs/public/submission.jsonl -q public_test/questions.jsonl

In [ ]:
# Copy file nộp ra /kaggle/working để tải về từ tab Output
!cp outputs/public/submission.jsonl /kaggle/working/submission.jsonl && ls -la /kaggle/working/submission.jsonl